### Artificial Intelligence 464.601 Project #7

### Before You Begin...
00. We're using a Jupyter Notebook environment (tutorial available here: https://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html),
01. Read the entire notebook before beginning your work, and
02.  Check the submission deadline on Gradescope.


### General Directions for this Assignment
00. Output format should be exactly as requested (it is your responsibility to make sure notebook looks as expected on Gradescope), and
01. Functions should do only one thing.


### Before You Submit...
00. Re-read the general instructions provided above, and
01. Submit your notebook (as .ipynb, not PDF) using Gradescope, and
02.  Do not submit any other files.

### Language Modeling

This project will require you to load and train models.  If you choose small models and datasets, you should be able to run this locally on your computer. However, larger models/datasets may require GPU access. You can access one GPU for free on [Google Colab](https://colab.research.google.com/).

We will use HuggingFace libraries. We discussed majority of what you will need during the discussion demo. Additional documentation can be found [here](https://huggingface.co/docs).

In [1]:
%pip install evaluate datasets transformers

  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached datasets-4.8.4-py3-none-any.whl.metadata (19 kB)
  Using cached transformers-5.6.2-py3-none-any.whl.metadata (33 kB)
  Using cached numpy-2.4.4-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 10.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 12.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 26.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 35.8 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 40.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 40.4 MB/s  0:00:00
Using cached numpy-2.4.4-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 MB 46.7 MB/s  0:00:01m0

In [2]:
# Imports
import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import pipeline
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments, Trainer
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, AutoModelForCausalLM
import os
os.environ["WANDB_DISABLED"] = "true"

/home/jason-lafita/miniconda3/envs/ai_hw7/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Problem 0: Data
From the [HuggingFace Datasets](https://huggingface.co/datasets), choose a dataset that satisfies the following criteria:
- Data must have train and test splits (Optional development set)
- Task must be text classification
- Task must have at least 3 labels


In [3]:
ds = load_dataset("dair-ai/emotion", "split")

Generating test split: 100%|██████████| 2000/2000 [00:00<00:00, 329055.35 examples/s]


**Describe the data.**
What is the utility of the task? What are the inputs? What are the labels? Are the any potential difficulties you expect from the task? How do you evaluate the performance of this task?

The inputs are a text statement in the form of a first-person statement. The labels are an integer, which corresponds to an emotion, such as 0 meaning "sadness" or 3 meaning "anger." The utility of this text is to train a model that can recognize human emotions.

The utility of this task is that a language model may be able to recognize human emotion, which is a very important aspect to language and communication. Tone and emotion words play a very large part in communication, and this can help a machine to accurately recognize and understand human speech. Also, in the face of people increasingly relying on chatbots for emotional support, it could be incredibly useful for an AI to understand when a speaker is in distress and point them towards apt resources.

I don't necessarily anticipate difficulties with this task itself. I think it will be difficult to train, since the sentences are quite longform and are different than the feature representations I've worked with in the mushroom dataset. However, I don't think that will be difficult to overcome, it will just take work and testing.

I will evaluate performance using the test dataset, to see if the model is able to anticipate emotion on data it has not seen. There are 6 classes, meaning random guesses would yield roughly 17% accuracy. The label split is as follows:

| Emotion | Int | Split |
| ------- | --- | ----- |
| 'joy' | 1 | 33.5% |
| 'sadness' | 0 | 29.5% |
| 'anger' | 3 | 13.5% |
| 'fear' | 4 | 12.1% |
| 'love' | 2 | 8.2% |
| 'surprise' | 5 | 3.6% |

An untrained model would most likely simply guess the highest-prevalence label, being 3: 'joy' at 33.5%. Therefore, I will evaluate the model to be performing a meaningful classification task if it is able to yield a response signfificantly above 33.5%. As a basic benchmark, I will shoot for 50% accuracy or above.

**Research current methods using this dataset.**
What is the current state of the art method? Describe the method, including the type of model used, training protocol (if any), and the performance. Cite your sources.

Lots of the models I've found on HF that use this dataset have been using BERT-like models. These models were quite vague in their descriptions, and the training protocol was a broken link. However, the results of the most popular model I saw using BERT was very promising, with a 0.938 test accuracy using BERT-like language models. This model was the most popular model trained on this dataset: https://huggingface.co/bhadresh-savani/distilbert-base-uncased-emotion. There were many other models that used BERT and the HF Trainer to yield similar results on the dataset.

Another model I found used transfer learning and looked quite promising. It uses a text-to-text transformer to classify text inputs on the emotion labels from the dataset. The transfer learning involved pre-training on a data-rich task, before fine-tuning the model on downstream tasks. This model produced an average f1-score of 0.83 across the dataset on the test data. https://huggingface.co/mrm8488/t5-base-finetuned-emotion

(Optional) If necessary, perform any data preprocessing here. For example, depending on the dataset you choose, you may need to clean the text or split the training set into a train and validation set.

In [5]:
# Preprocessing and cleaning already completed

{'text': ['i didnt feel humiliated', 'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake', 'im grabbing a minute to post i feel greedy wrong', 'i am ever feeling nostalgic about the fireplace i will know that it is still on the property', 'i am feeling grouchy'], 'label': [0, 0, 3, 2, 3]}


### Problem 1: Fine-Tuning a Model
Choose an encoder-only model. Load the model and add a classification layer.

Describe the model you choose. What are the unique properties of this model? What are the pros and cons? Cite your sources.

<font color="red">TODO</font>

Finetune the model on your dataset. Report the performance on the test set.

In [ ]:
# TODO

### Problem 2: Error Analysis

Conduct an error analysis on your models. What are your models good at? What do they get wrong? Provide examples of both correct and incorrect predictions. Suggest methods to improve the performance.

In [ ]:
# TODO

## Before You Submit...

00. Re-read the general instructions provided above, and
01. Hit "Kernel"->"Restart & Run All". The first cell that is run should show [1], the second should show [2], and so on...
02. Submit your notebook (as .ipynb, not PDF) using Gradescope, and
03.  Do not submit any other files.